# Fase 1: Data Infestion & Cleaning

In [40]:
# --- FASE 1: Data Ingestion & Cleaning ---
import pandas as pd
import numpy as np

# Percorsi dei file
file_path_storico = "./dataset/Archivio con lo storico prezzi dei titoli in formato CSV(Ultimo aggiornamento 07-05-2026).csv"
file_path_anagrafica = "./dataset/Obbligazioni quotate su Borsa Italiana(Ultimo aggiornamento 07-05-2026).csv"

# Caricamento storico prezzi 
df_storico = pd.read_csv(file_path_storico,
                        sep=';',
                        decimal=',',
                        parse_dates=['referencedate', 'endvaluedate'],
                        dayfirst=True)

# Caricamento anagrafica obbligazioni
try:
    df_anagrafica = pd.read_csv(file_path_anagrafica, sep=';', decimal=',')
    print(f"Anagrafica caricata: {df_anagrafica.shape[0]} righe")
except Exception as e:
    print("Errore nel caricamento dell'anagrafica:", e)
    df_anagrafica = None

# Pulizia base: rimozione spazi, uniformità nomi colonne
if df_anagrafica is not None:
    df_anagrafica.columns = df_anagrafica.columns.str.strip().str.lower()
    df_storico.columns = df_storico.columns.str.strip().str.lower()

# Cast di alcune colonne (esempio: rating a stringa, isin a stringa)
if df_anagrafica is not None:
    if 'rating' in df_anagrafica.columns:
        df_anagrafica['rating'] = df_anagrafica['rating'].astype(str)
    if 'isincode' in df_anagrafica.columns:
        df_anagrafica['isincode'] = df_anagrafica['isincode'].astype(str)
    if 'isincode' in df_storico.columns:
        df_storico['isincode'] = df_storico['isincode'].astype(str)

# Visualizzazione di controllo
print("\nDimensioni storico prezzi:", df_storico.shape)
if df_anagrafica is not None:
    print("Dimensioni anagrafica:", df_anagrafica.shape)
    display(df_anagrafica.head())
display(df_storico.head())

Anagrafica caricata: 3629 righe

Dimensioni storico prezzi: (827771, 10)
Dimensioni anagrafica: (3629, 13)


,firstdate,marketcode,segmentcode,isincode,description,currencycode,minimumlot,status,issuedate,issueprice,redemptiondate,redemptionprice,unnamed: 12
0,07/05/2026,EUROTLX,EUROTLX,XS3318840223,Lottomaticag Fx 4.625% Apr32 Call Eur,EUR,100000.0,NaN,NaN,NaN,30/04/2032,NaN,NaN
1,07/05/2026,EUROTLX,EUROTLX,XS2433136194,Israel Fx 0.625% Jan32 Eur,EUR,1000.0,NaN,NaN,NaN,18/01/2032,NaN,NaN
2,07/05/2026,EUROTLX,EUROTLX,XS1936101291,Israel Fx 2.5% Jan49 Eur,EUR,1000.0,NaN,NaN,NaN,16/01/2049,NaN,NaN
3,07/05/2026,EUROTLX,EUROTLX,XS1936100483,Israel Fx 1.5% Jan29 Eur,EUR,1000.0,NaN,NaN,NaN,16/01/2029,NaN,NaN
4,07/05/2026,EUROTLX,EUROTLX,US46513JB593,Israel Fx 4.5% Apr20 Usd,USD,200000.0,NaN,NaN,NaN,03/04/2120,NaN,NaN


,isincode,marketcode,referencedate,endvaluedate,pricetype,pricevalue,volume,mintoday,maxtoday,unnamed: 9
0,AT0000383864,MOT,2023-01-02,2023-01-03,RP,115.22,0,0.00,0.00,NaN
1,AT0000383864,MOT,2023-01-03,2023-01-04,RP,115.59,0,0.00,0.00,NaN
2,AT0000383864,MOT,2023-01-04,2023-01-05,LP,116.35,4000,116.35,116.38,NaN
3,AT0000383864,MOT,2023-01-05,2023-01-06,LP,116.50,5000,116.50,116.50,NaN
4,AT0000383864,MOT,2023-01-06,2023-01-09,LP,116.60,2000,116.60,116.60,NaN


In [41]:
# Estraiamo gli ISIN unici sotto forma di "Insiemi" (Set)
isins_in_anagrafica = set(df_anagrafica['isincode'].unique())
isins_in_storico = set(df_storico['isincode'].unique())

# Analisi dell'intersezione (Il KDD process)
common_isins = isins_in_anagrafica.intersection(isins_in_storico)
only_in_anagrafica = isins_in_anagrafica - isins_in_storico
only_in_storico = isins_in_storico - isins_in_anagrafica

# Stampiamo il report
print(f"--- REPORT DATA CONSISTENCY ---")
print(f"Totale ISIN unici in Anagrafica: {len(isins_in_anagrafica)}")
print(f"Totale ISIN unici in Storico:    {len(isins_in_storico)}")
print(f"ISIN COMUNI (Utilizzabili per ML): {len(common_isins)}")
print(f"-------------------------------")
print(f"ISIN scartati (Solo in Anagrafica, no prezzi): {len(only_in_anagrafica)}")
print(f"ISIN scartati (Solo in Storico, no anagrafica): {len(only_in_storico)}")

--- REPORT DATA CONSISTENCY ---
Totale ISIN unici in Anagrafica: 3504
Totale ISIN unici in Storico:    1309
ISIN COMUNI (Utilizzabili per ML): 1309
-------------------------------
ISIN scartati (Solo in Anagrafica, no prezzi): 2195
ISIN scartati (Solo in Storico, no anagrafica): 0


In [42]:
# Il filtraggio vero e proprio
# La funzione .isin() di Pandas è ottimizzata per questo task.
# Controlla se il valore nella colonna 'isincode' è presente nella nostra lista di ISIN comuni.
df_anagrafica_filtrato = df_anagrafica[df_anagrafica['isincode'].isin(common_isins)].copy()
df_storico_filtrato = df_storico[df_storico['isincode'].isin(common_isins)].copy()


In [43]:
# Carichiamo solo ISIN e Data per fare velocissimi
# parse_dates e dayfirst=True sono fondamentali per fargli capire che è GG/MM/AAAA
df_dates = pd.read_csv(file_path_storico, 
                       sep=';', 
                       usecols=['isincode', 'referencedate'],
                       parse_dates=['referencedate'], 
                       dayfirst=True)

# 1. Estrazione del Time Span Globale
min_date = df_dates['referencedate'].min()
max_date = df_dates['referencedate'].max()

print(f"\n--- TIME SPAN DEL DATASET STORICO ---")
print(f"Data assoluta più vecchia: {min_date.strftime('%d/%m/%Y')}")
print(f"Data assoluta più recente: {max_date.strftime('%d/%m/%Y')}")
print(f"Profondità totale in giorni: {(max_date - min_date).days} giorni")

# 2. Analisi profonda (Quanti dati ha davvero ogni obbligazione?)
# Raggruppiamo per ISIN e calcoliamo quanti giorni di dati ha ciascuno
history_lengths = df_dates.groupby('isincode')['referencedate'].count()

print(f"\n--- ANALISI STORICO PER SINGOLO TITOLO ---")
print(f"Giorni di storico MEDIO per bond: {history_lengths.mean():.0f}")
print(f"Giorni di storico MEDIANO: {history_lengths.median():.0f}")
print(f"Titoli con meno di 6 mesi di dati (< 130 giorni borsa): {(history_lengths < 130).sum()}")
print(f"Titoli con più di 1 anno di dati (> 250 giorni borsa): {(history_lengths > 250).sum()}")


--- TIME SPAN DEL DATASET STORICO ---
Data assoluta più vecchia: 02/01/2023
Data assoluta più recente: 07/05/2026
Profondità totale in giorni: 1221 giorni

--- ANALISI STORICO PER SINGOLO TITOLO ---
Giorni di storico MEDIO per bond: 632
Giorni di storico MEDIANO: 833
Titoli con meno di 6 mesi di dati (< 130 giorni borsa): 120
Titoli con più di 1 anno di dati (> 250 giorni borsa): 1109


A questo punto mi sono ricordato che probabilmente tra queste obbligazioni ce ne sono alcune non standard e quindi dobbiamo escludere:
1.  **Obbligazioni non governative e non europee:** Aziendali (corporate), bancarie, ecc. hanno un rischio di credito diverso che "sporcherebbe" il modello.
2.  **Obbligazioni "esotiche":** Quelle con cedole variabili (step-up, inflation-linked come i BTP Italia/Futura) hanno un modello di prezzo completamente diverso e non possono essere confrontate con i bond a tasso fisso.

In [44]:
# --- FASE 1.5: FILTRAGGIO AVANZATO DELL'UNIVERSO DI TRADING ---

# Lavoriamo sul DataFrame 'df_anagrafica_filtrato' che contiene già i 1309 ISIN comuni
print(f"ISIN totali prima del filtraggio avanzato: {df_anagrafica_filtrato['isincode'].nunique()}")

# 1. Filtro per Obbligazioni Governative Europee
# Usiamo il codice ISIN che inizia con il prefisso del paese (IT, DE, FR, etc.)
# e la colonna 'issuer' che di solito contiene 'GOV' per i governativi.
european_gov_codes = ['IT', 'DE', 'FR', 'ES', 'PT', 'AT', 'BE', 'NL', 'FI', 'IE', 'GR']

# Creiamo una maschera booleana: ISIN deve iniziare con uno dei codici E l'emittente deve essere governativo
is_european_gov = (df_anagrafica_filtrato['isincode'].str.startswith(tuple(european_gov_codes))) & \
                  (df_anagrafica_filtrato['description'].str.contains('GOV|Btp|Schatz|OAT|Bonos|Obligaciones|Bund|BOT|CTZ|Bubill|Zc', case=False, na=False))

df_anagrafica_gov = df_anagrafica_filtrato[is_european_gov].copy()
print(f"ISIN rimasti dopo filtro 'EU Government': {df_anagrafica_gov['isincode'].nunique()}")

# 2. Filtro per escludere Obbligazioni "Esotiche" (a cedola non fissa)
# Cerchiamo parole chiave nella descrizione del titolo.
parole_chiave_escluse = ['Futura', 'Italia', 'Inflazione', 'Indicizzato', 'Inflation', 'Valore', 'Btpi', 'Linker', "Piu"]

pattern_esclusione = '|'.join(parole_chiave_escluse)

# Creiamo una maschera per trovare le righe che contengono queste parole (case=False per ignorare maiuscole/minuscole)
is_exotic = df_anagrafica_gov['description'].str.contains(pattern_esclusione, case=False, na=False)

# Usiamo l'operatore tilde (~) per invertire la maschera, ovvero "TENIAMI TUTTO QUELLO CHE NON È ESOTICO"
df_anagrafica_clean = df_anagrafica_gov[~is_exotic].copy()
print(f"ISIN rimasti dopo filtro 'Esotici': {df_anagrafica_clean['isincode'].nunique()}")

# 3. Otteniamo la lista finale di ISIN "puri" su cui lavorare
final_clean_isins = set(df_anagrafica_clean['isincode'].unique())

print("\n--- RISULTATO FINALE ---")
print(f"Numero di obbligazioni 'Plain Vanilla' selezionate per il modello: {len(final_clean_isins)}")

# ORA POSSIAMO FILTRARE LO STORICO CON LA NUOVA LISTA DI ISIN PULITI
df_storico_clean = df_storico_filtrato[df_storico_filtrato['isincode'].isin(final_clean_isins)].copy()

print(f"Dimensioni Storico finale (pulito): {df_storico_clean.shape}")


ISIN totali prima del filtraggio avanzato: 1309
ISIN rimasti dopo filtro 'EU Government': 337
ISIN rimasti dopo filtro 'Esotici': 305

--- RISULTATO FINALE ---
Numero di obbligazioni 'Plain Vanilla' selezionate per il modello: 305
Dimensioni Storico finale (pulito): (189510, 10)


# Fase 2: Feature Engeneering

In [45]:
# --- CONTROLLO QUALITÀ DATI (Best Practice) ---
# 1. Duplicati su ISIN+Data
if 'isincode' in df_storico_filtrato.columns and 'referencedate' in df_storico_filtrato.columns:
    n_dupes = df_storico_filtrato.duplicated(subset=['isincode', 'referencedate']).sum()
    print(f"Duplicati su ISIN+Data nello storico filtrato: {n_dupes}")
    
# 2. Missing nelle colonne chiave
print("Missing values nelle colonne chiave dello storico:")
print(df_storico_filtrato[['isincode', 'referencedate']].isnull().sum())

if 'isincode' in df_anagrafica_filtrato.columns:
    print("Missing values nelle colonne chiave dell'anagrafica:")
    print(df_anagrafica_filtrato['isincode'].isnull().sum())

Duplicati su ISIN+Data nello storico filtrato: 8266
Missing values nelle colonne chiave dello storico:
isincode         0
referencedate    0
dtype: int64
Missing values nelle colonne chiave dell'anagrafica:
0


In [46]:
# --- ESTRAZIONE DEL COUPON ---
import re

def extract_coupon(description):
    if pd.isnull(description):
        return np.nan
    desc = str(description)

    # GESTIONE ZERO COUPON: Se è un BOT o contiene "zc" (zero coupon), la cedola è 0% matematico
    if 'bot' in desc or 'Zc' in desc or 'zero' in desc or 'ctz' in desc:
        return 0.0

    # Cerchiamo pattern di coupon: numeri seguiti da % (es. "5%", "3,5%")
    match = re.search(r'(\d+[\.,]\d+|\d+)[ ]*%', desc)
    if match:
        return float(match.group(1).replace(',', '.'))
    # Cerchiamo pattern di coupon: numeri preceduti da EUR
    match = re.search(r'[Ee][Uu][Rr][ ]*(\d+[\.,]\d+|\d+)', desc)
    if match:
        return float(match.group(1).replace(',', '.'))   
    return np.nan

df_anagrafica_clean['coupon'] = df_anagrafica_clean['description'].apply(extract_coupon)
# Verfichiamo i risultati dell'estrazione del coupon
print(df_anagrafica_clean[['description', 'coupon']].head(10))
print(df_anagrafica_clean['coupon'].describe())
perc_nan = df_anagrafica_clean['coupon'].isnull().mean() * 100
print(f"Percentuale di coupon NaN: {perc_nan:.2f}%")
missing_coupon = df_anagrafica_clean[df_anagrafica_clean['coupon'].isnull()]
print(f"Obbligazioni con coupon non estratto: {missing_coupon.shape[0]}")
display(missing_coupon[['description']].head(10))

# scarto l'unico che di cui non riesco ad estrarre il coupon (è un titolo "esotico" che è sfuggito al filtro precedente)
df_anagrafica_clean = df_anagrafica_clean.dropna(subset=['coupon']).copy()

                         description  coupon
17            Btp Fx 3.15% Jun31 Eur    3.15
19          Schatz Fx 2.5% Jun28 Eur    2.50
34             Btp Fx 3.8% Jul36 Eur    3.80
61             Btp Fx 3.3% Jun33 Eur    3.30
64                Bot Zc Apr27 A Eur    0.00
65                Bot Zc Jul26 Q Eur    0.00
97                Bot Zc Sep26 S Eur    0.00
168               Bot Zc Mar27 A Eur    0.00
181  Obligaciones Fx 3.95% Oct56 Eur    3.95
199          Bonos Fx 2.6% May31 Eur    2.60
count    304.000000
mean       2.335609
std        1.660928
min        0.000000
25%        0.850000
50%        2.500000
75%        3.450000
max        7.250000
Name: coupon, dtype: float64
Percentuale di coupon NaN: 0.33%
Obbligazioni con coupon non estratto: 1


,description
2927,Mediolomb-98/28 25zc


In [49]:
# --- CALCOLO DAYS TO MATURITY (DTM) ---
# Uniamo le informazioni fisse (cedola, scadenza) al file dei prezzi giornalieri
df_ml = pd.merge(df_storico_clean, 
                 df_anagrafica_clean[['isincode', 'description', 'redemptiondate', 'coupon']], 
                 on='isincode', 
                 how='inner')

# DTM = Differenza in giorni tra la Data di Scadenza e la Data in cui è stato registrato il Prezzo
df_ml['redemptiondate'] = pd.to_datetime(df_ml['redemptiondate'], dayfirst=True, errors='coerce')
df_ml['days_to_maturity'] = (df_ml['redemptiondate'] - df_ml['referencedate']).dt.days

# A volte alcuni provider mantengono i prezzi anche dopo la scadenza, vogliamo evitare di avere dati "sporchi" con DTM negativi
# Scartiamo le righe dove il bond è già scaduto (DTM <= 0)
df_ml = df_ml[df_ml['days_to_maturity'] > 0].copy()

# Aggiungiamo anche gli ANNI alla scadenza (Years to Maturity), molto usati in finanza
df_ml['years_to_maturity'] = df_ml['days_to_maturity'] / 365.25

print(df_ml[['isincode', 'referencedate', 'pricevalue', 'coupon', 'years_to_maturity']].head(10))

       isincode referencedate  pricevalue  coupon  years_to_maturity
0  DE0001030708    2023-01-02       83.82     0.0           7.616701
1  DE0001030708    2023-01-03       84.07     0.0           7.613963
2  DE0001030708    2023-01-04       84.62     0.0           7.611225
3  DE0001030708    2023-01-05       84.37     0.0           7.608487
4  DE0001030708    2023-01-06       85.00     0.0           7.605749
5  DE0001030708    2023-01-09       84.98     0.0           7.597536
6  DE0001030708    2023-01-10       84.59     0.0           7.594798
7  DE0001030708    2023-01-11       85.30     0.0           7.592060
8  DE0001030708    2023-01-12       85.67     0.0           7.589322
9  DE0001030708    2023-01-13       85.78     0.0           7.586585
